# Class 25: Tool Calling & Autonomous Logic
Model-কে "hands" দেওয়া: model শুধু অনুরোধ করে, চালায় আমাদের Python। এই notebook-এ tool লিখব, agent loop বানাব, আর sandbox-এর নিরাপত্তা পরীক্ষা করব। Ollama ছাড়াই চলে এমন cell আগে, LLM cell শেষে।

## 1. Tool হলো সাধারণ Python function
আগে safe calculator লিখি। `eval()` নয়, `ast` whitelist।

In [ ]:
import ast, math, operator
OPS={ast.Add:operator.add,ast.Sub:operator.sub,ast.Mult:operator.mul,ast.Div:operator.truediv,ast.Pow:operator.pow,ast.USub:operator.neg}
def _ev(n):
    if isinstance(n,ast.Constant) and isinstance(n.value,(int,float)): return n.value
    if isinstance(n,ast.BinOp) and type(n.op) in OPS:
        l,r=_ev(n.left),_ev(n.right)
        if isinstance(n.op,ast.Pow) and abs(r)>100: raise ValueError("exponent too large")
        return OPS[type(n.op)](l,r)
    if isinstance(n,ast.UnaryOp) and type(n.op) in OPS: return OPS[type(n.op)](_ev(n.operand))
    if isinstance(n,ast.Call) and isinstance(n.func,ast.Name) and n.func.id=="sqrt": return math.sqrt(*[_ev(a) for a in n.args])
    raise ValueError("unsupported expression")
def calculate(expression):
    try: return str(_ev(ast.parse(expression,mode="eval").body))
    except Exception as e: return f"Error: {e}"
for e in ["(12+8)*3","sqrt(144)","__import__('os').system('dir')","9**9**9"]:
    print(e,"->",calculate(e))

## 2. Sandboxed file tool
`resolve()` করে দেখি result sandbox-এর ভেতরে কিনা। শুধু ".." খোঁজা যথেষ্ট না।

In [ ]:
import tempfile
from pathlib import Path
box=Path(tempfile.mkdtemp())/"box"; box.mkdir()
(box/"notes.txt").write_text("The launch code is 4471.")
(box.parent/"secret.txt").write_text("outside!")
def read_file(path):
    root=box.resolve(); t=(root/path).resolve()
    if not t.is_relative_to(root): return "Error: path escapes sandbox"
    return t.read_text() if t.is_file() else "Error: file not found"
for p in ["notes.txt","../secret.txt",str(box.parent/"secret.txt")]:
    print(p,"->",read_file(p))

## 3. Tool schema = prompt
Model শুধু এই JSON দেখে। `description` ই ঠিক করে সে কখন tool ডাকবে।

In [ ]:
TOOLS=[{"type":"function","function":{"name":"calculate","description":"Evaluate a math expression exactly. Use for ANY arithmetic.",
 "parameters":{"type":"object","properties":{"expression":{"type":"string"}},"required":["expression"]}}},
 {"type":"function","function":{"name":"read_file","description":"Read a text file inside the sandbox.",
 "parameters":{"type":"object","properties":{"path":{"type":"string"}},"required":["path"]}}}]
REGISTRY={"calculate":lambda expression:calculate(expression),"read_file":lambda path:read_file(path)}
def run_tool(name,args):
    try: return REGISTRY[name](**args)
    except KeyError: return f"Error: unknown tool {name}"
    except TypeError as e: return f"Error: bad arguments ({e})"
print(run_tool("calculate",{"expression":"2**10"}), run_tool("nope",{}), run_tool("read_file",{}))

## 4. Agent loop (fake model দিয়ে, Ollama লাগে না)
Loop-এর যুক্তি আলাদা করে বোঝার জন্য model-কে scripted function দিয়ে বদলাই। থামার শর্ত দুটো: tool call নেই, অথবা `max_steps` শেষ।

In [ ]:
def run_agent(question, chat_fn, max_steps=5):
    messages=[{"role":"system","content":"Use tools for math and files."},{"role":"user","content":question}]
    trace=[]
    for _ in range(max_steps):
        reply=chat_fn(messages,TOOLS)
        if not reply["tool_calls"]: return reply["content"],trace
        messages.append({"role":"assistant","content":reply["content"],"tool_calls":[{"function":c} for c in reply["tool_calls"]]})
        for c in reply["tool_calls"]:
            r=run_tool(c["name"],c["arguments"]); trace.append((c["name"],c["arguments"],r))
            messages.append({"role":"tool","content":r,"tool_name":c["name"]})
    return "Stopped: max_steps reached",trace
script=iter([{"content":"","tool_calls":[{"name":"calculate","arguments":{"expression":"6*7"}}]},{"content":"42","tool_calls":[]}])
print(run_agent("6*7?",lambda m,t:next(script)))
runaway=lambda m,t:{"content":"","tool_calls":[{"name":"calculate","arguments":{"expression":"1+1"}}]}
print(run_agent("loop",runaway,max_steps=3)[0])

## 5. Real model: Ollama
**Model reference table**

| Model | Use Case | Advantage | Limitation |
| --- | --- | --- | --- |
| `llama3.1:8b` (chosen) | Local tool calling, agents | Native tool support, argument ভালো | ~5GB RAM, CPU-তে ধীর |
| `mistral:7b` | General chat + tool calling | দ্রুত, instruction following ভালো | Multi-step-এ tool call মাঝে মাঝে বাদ পড়ে |
| `llama3.2:3b` | Low-RAM laptop, দ্রুত demo | ~2GB, দ্রুত | জটিল বা একাধিক tool-এ নির্ভরযোগ্যতা কম |

আগে `ollama serve` আর `ollama pull llama3.1:8b`। Server বন্ধ থাকলে নিচের cell friendly message দেখায়।

In [ ]:
def ollama_chat(model):
    def fn(messages,tools):
        import ollama
        m=ollama.Client(host="http://localhost:11434").chat(model=model,messages=messages,tools=tools,options={"temperature":0})["message"]
        return {"content":m.get("content") or "","tool_calls":[{"name":c["function"]["name"],"arguments":dict(c["function"].get("arguments") or {})} for c in (m.get("tool_calls") or [])]}
    return fn
QUESTION="What is (12+8)*3 divided by 4?"
for model in ["llama3.1:8b","mistral:7b","llama3.2:3b"]:
    try: print(model,"->",run_agent(QUESTION,ollama_chat(model)))
    except Exception as e: print(model,"-> Ollama unavailable or model not pulled:",type(e).__name__)

## 6. Brain Teaser
১. `max_steps` না থাকলে কী ঘটত? ২. Tool error exception না হয়ে string কেন? ৩. তিনটা model-এর trace তুলনা করুন: কোনটা tool বাদ দিয়ে নিজে অঙ্ক কষল?